# To COLAB notebook

The GPU did not like maths a lot and is on strike until further notice. Computer crashes within an hour after starting to train on it.

This notebook is not really intended to be part of any pipeline, it's more like a makeshift solution for a problem that shouldn't exist. Colab is limited in space; we cannot upload all the data at once. At least for step 3.3 we can try to zip brands per angle and upload it to Colab. This upload will happen angle per angle and then training happens one upload at a time. google Colab is too small to do the full set, but it might work for just the individual angles. So no code will be added to shared utilities as this is mostly a self-contained issue.

To further reduce storage requirements and cloud compute times we'll apply extra pre-processing in this notebook: 
- compress augmented images
- pre-crop all images to their bounding boxes (step 3.2 proved that cropping helped - so we continue with this.)

In [1]:
import pandas as pd
import os
import sys
sys.path.append('../utils')
import config_handling as conf
import cnn_helpers
import numpy as np
from PIL import Image


2025-03-22 08:05:05.862492: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
SHAPE = 224
config = conf.read_config('../config/automotive.conf.ini')
basedir = config['settings']['image_directory']

augment_base = os.path.join(basedir, 'augmentated data')
augment_csv_dump = os.path.join(basedir, 'CSV-data')

In [6]:
phases = os.listdir(augment_csv_dump)
phases

for i, phase in enumerate(phases):
    print(f"type {i} for {phase}")

type 0 for brand phase


In [7]:
phase = input()
try:
    phase = phases[int(phase)]
except:
    raise Exception("You chose poorly")

print(f'CHOSEN: {phase}')

CHOSEN: brand phase


read CSV data for the chosen phase

In [8]:
os.listdir(os.path.join(augment_csv_dump, phase))

['validationdata_brandphase.csv',
 'testdata_brandphase.csv',
 'traindata_brandphase.csv']

In [9]:
train_df = pd.read_csv(os.path.join(augment_csv_dump, phase, 'traindata_brandphase.csv'))
test_df = pd.read_csv(os.path.join(augment_csv_dump, phase, 'testdata_brandphase.csv'))

In [ ]:
def shrink(image_object, w, h):
    """resizes an image object read by PIL to given wh dimension
    image_object = Object by PIL
    w = int = width target
    h = int = height target
    returns a PIL image object;
    """
    return image_object.resize((round(w), round(h)), Image.LANCZOS)
    

def crop_image(path, x1, y1, x2, y2): 
    """ crops a given image referenced by it's path to boundingbox
    defined by topleft (x1, y2) and bottomright (x2, y2)
    
    path = str = Absolute path to an image
    x1 = int = topleft x-coord
    y1 = int = topleft y-coord
    x2 = int = bottomright x-coord
    y2 = int = bottomright y-coord.
    """
    with Image.open(path) as img:
        if x1 >= 0:
            return img.crop((x1, y1, x2, y2))
        else:
            w,h = img.size
            return img.crop((0,0,w,h))
        
        

def process_subset(df, target_base, reduction_key):
    for key, small_df in df.groupby(reduction_key): 
        csv_content = []
        if key in os.listdir(target_base):
            print('skipping ', key)
            continue
        subdir = os.path.join(target_base, key)

        for brand, brand_df in small_df.groupby('brand'):
            os.makedirs(os.path.join(subdir, 'augmented', brand))
            os.makedirs(os.path.join(subdir, 'original', brand))
            for _, row in brand_df.iterrows():
                image_path = row['abs_path']
                x1 = int(row['yolobox_top_left_x'])
                y1 = int(row['yolobox_top_left_y'])
                x2 = int(row['yolobox_bottom_right_x'])
                y2 = int(row['yolobox_bottom_right_y'])
                #crop
                image = crop_image(image_path, x1, y1, x2, y2)
                #resize so that shortest side is 224 (keep aspect ratio locked)
                w,h = image.size
                shortest = np.min([w,h])
                if shortest > SHAPE: 
                    factor = shortest/SHAPE
                    image = shrink(image, w/factor, h/factor)
                name = image_path.split('/')[-1]
                if '/augmentated data/' in image_path: 
                    destination = os.path.join(subdir, 'augmented', brand, name)
                else:
                    destination = os.path.join(subdir, 'original', brand, name)
                image.save(destination)
                csv_content.append([brand, key, destination])
        df_shrunk = pd.DataFrame(csv_content)
        df_shrunk.to_csv(os.path.join(target_base, key, 'datafile.csv'))

In [11]:
process_subset(train_df, '/home/frederic/Downloads/traindata', 'model_label')


skipping  front
skipping  frontleft
skipping  frontright


In [ ]:
process_subset(test_df, '/home/frederic/Downloads/testdata', 'model_label')
